# PV2H 演示：数据 → 训练 → 预测 → 曲线 → 南网回测

本 Notebook 用于**展示运行流程**（赛题为 **3 个场站** S01–S03；下表汇总行为第 4 行「三站等权」）。

南网两项评价指标（`TASK.md` §6.3，`tools/evaluate.py` 输出）：
1. **场站南网日均准确率** \(A_s\)
2. **三站等权南网日均准确率** \(\mathrm{Score}\)

运行前：在仓库根目录将 `DATA_ROOT` / `PV2H_DATA` 指向含 `cloud/`、`power/`、`tools/` 的数据包。**推荐**在项目下创建虚拟环境并选为 Jupyter 内核：`python -m venv .venv && .venv/bin/pip install -r requirements.txt`。首格会检测缺包及 pyarrow 二进制不兼容并尝试自动修复。

In [ ]:
from pathlib import Path
import os
import sys
import json
import importlib.util
import subprocess
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("pv2h.demo")

import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "predict.py").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PINNED_PARQUET = ("pyarrow==25.0.1", "pandas==2.3.3")


def _purge_modules(prefixes: tuple[str, ...]) -> None:
    for name in list(sys.modules):
        if any(name == p or name.startswith(p + ".") for p in prefixes):
            del sys.modules[name]


def _reinstall_parquet_stack() -> None:
    logger.info("重装 pyarrow/pandas（修复 C 扩展二进制不兼容）…")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", *PINNED_PARQUET],
    )
    _purge_modules(("pyarrow", "pandas"))


def _parquet_stack_ok() -> bool:
    try:
        import pyarrow as pa

        pa.array([1, 2, 3])
        import pyarrow.parquet  # noqa: F401
        return True
    except Exception:
        return False


def _ensure_requirements() -> None:
    """Notebook 内核常与终端 venv 不一致；缺包或 pyarrow 损坏时自动修复。"""
    need = ("pvlib", "lightgbm", "pyarrow", "sklearn")
    missing = [name for name in need if importlib.util.find_spec(name) is None]
    if missing:
        logger.info("当前内核缺少: %s → pip install -r requirements.txt …", ", ".join(missing))
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")],
        )
        _purge_modules(("pyarrow", "pandas"))
    if not _parquet_stack_ok():
        _reinstall_parquet_stack()
    if not _parquet_stack_ok():
        raise RuntimeError(
            "pyarrow 仍不可用。请在终端执行:\n"
            f"  {sys.executable} -m pip install --force-reinstall pyarrow==25.0.1 pandas==2.3.3\n"
            "然后 Kernel → Restart，再 Run All。更推荐: 创建 .venv 并选其为 Jupyter 内核。"
        )


_ensure_requirements()
import pandas as pd


def read_samples_parquet(path: Path) -> pd.DataFrame:
    try:
        return pd.read_parquet(path)
    except ValueError as exc:
        msg = str(exc)
        if "Array size changed" not in msg and "binary incompatibility" not in msg.lower():
            raise
        _reinstall_parquet_stack()
        import pandas as pd_local

        return pd_local.read_parquet(path)


from pv2h_model import config
from pv2h_model.data import load_stations, make_release
from pv2h_model.eval.reporting import (
    setup_matplotlib_cjk,
    curve_frame,
    nanwang_report_table,
    official_score,
    predictions_from_samples,
    score_rows_metric,
)
from predict import predict, get_predictor

setup_matplotlib_cjk()

# ---------- 可改参数 ----------
DATA_ROOT = Path(os.environ.get("PV2H_DATA", config.DATA_ROOT))
SPLIT = "val"  # train | val

# 训练：首次演示建议 False（使用仓库自带 models/）；需要重训时改 True
RUN_TRAIN = False
TRAIN_STAGES = "stats,samples"  # 不含 members（耗时数小时）；全量: "all"

# 在线起报演示（慢，读云图 ZIP）
DEMO_ISSUE = "09-01 02:00"

# 绘图：目标功率时刻区间（UTC MM-DD HH:mm）
PLOT_STATION = "S01"
PLOT_LEAD = 60
PLOT_TARGET_START = "09-01 08:00"
PLOT_TARGET_END = "09-01 18:00"

logger.info("ROOT: %s", ROOT)
logger.info("DATA_ROOT: %s", DATA_ROOT)
logger.info(
    "models: %s blend: %s",
    config.MODELS_DIR.exists(),
    (config.MODELS_DIR / "blend.json").exists(),
)

当前内核缺少: pvlib → pip install -r requirements.txt …


## 1. 数据读取

In [ ]:
stations = load_stations()
power = pd.read_csv(
    DATA_ROOT / "power" / f"{SPLIT}.csv.gz",
    dtype={"station_id": str},
    float_precision="round_trip",
)
logger.info("场站:\n%s", stations)
logger.info(
    "功率记录: %s 时间范围: %s ~ %s",
    power.shape,
    power.timestamp.min(),
    power.timestamp.max(),
)

with make_release(DATA_ROOT) as ds:
    demo_in = ds.inputs(
        DEMO_ISSUE,
        power_steps=config.POWER_STEPS,
        cloud_frames=config.CLOUD_FRAMES,
        bands=config.CLOUD_BANDS,
    )
logger.info(
    "起报: %s 功率窗口: %s 云帧: %s",
    DEMO_ISSUE,
    demo_in["power_ratio"].shape,
    next(iter(demo_in["cloud"].values())).shape,
)

## 2. 训练（可选）

默认使用 Git 仓库中的 `models/`。若 `RUN_TRAIN=True`，调用 `train.py` 按阶段重建统计量 / 样本 / 成员。

In [ ]:
if RUN_TRAIN:
    for stage in TRAIN_STAGES.split(","):
        stage = stage.strip()
        logger.info("=== train stage: %s ===", stage)
        subprocess.run(
            [sys.executable, str(ROOT / "train.py"), "--stage", stage],
            cwd=str(ROOT),
            check=True,
        )
else:
    logger.info("跳过训练，加载已有模型: %s", json.loads((config.MODELS_DIR / "blend.json").read_text())["members"])

bp = get_predictor()
logger.info("融合成员: %s", bp.members)

## 3. 预测

- **在线**：单次起报 `predict(inputs)`（与提交包一致）
- **离线快路径**：在存在 `cache/samples_{split}.parquet` 时，对整表特征一次融合（用于回测与绘图）

In [ ]:
online_pred = predict(demo_in)
display(online_pred.head(8))

samples_path = config.CACHE_DIR / f"samples_{SPLIT}.parquet"
if not samples_path.exists():
    raise FileNotFoundError(
        f"缺少 {samples_path}，请先 RUN_TRAIN 且 TRAIN_STAGES 含 samples，或运行: python train.py --stage samples"
    )

samples = read_samples_parquet(samples_path)
offline_pred = bp.predict_frame(samples)
logger.info("离线样本预测: %s 行", len(offline_pred))

## 4. 预测 vs 实测曲线（可设定目标时刻区间）

固定 `lead=60`：横轴为**目标时刻**，每 15 分钟一点来自不同起报。首格已配置 matplotlib 中文字体（避免标题乱码）。

In [ ]:
curve = curve_frame(
    samples,
    offline_pred,
    station_id=PLOT_STATION,
    lead_minutes=PLOT_LEAD,
    target_start=PLOT_TARGET_START,
    target_end=PLOT_TARGET_END,
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(curve["target_time"], curve["y_true"], label="实测 power_ratio", linewidth=1.2)
ax.plot(curve["target_time"], curve["pred"], label=f"预测 lead={PLOT_LEAD}min", linewidth=1.2, alpha=0.85)
if "clearsky_target" in curve.columns:
    ax.plot(curve["target_time"], curve["clearsky_target"], label="clearsky_target", linestyle="--", alpha=0.5)
ax.set_title(f"{PLOT_STATION} 目标时刻 {PLOT_TARGET_START} ~ {PLOT_TARGET_END} (UTC)")
ax.set_xlabel("target_time (UTC)")
ax.set_ylabel("capacity-normalized power")
ax.legend(loc="upper right")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
logger.info(
    "曲线有效点数: %s（仅 is_missing=0 的实测；午后近 0 为真实低出力，不是表格里的占位空值）",
    len(curve),
)

## 5. 回测：南网精度（官方 + 行级对照）

生成完整验证集预测表并调用 `tools/evaluate.py` 同口径打分。表中「—」表示该指标不适用于该行（场站行无 Score，汇总行无单站 A_s），不是缺失数据。

In [ ]:
pred_df = predictions_from_samples(samples, offline_pred)
official = official_score(pred_df, DATA_ROOT, split=SPLIT)
rows_metric = score_rows_metric(samples, offline_pred)

report = nanwang_report_table(official)
display(report)

logger.info("官方 JSON（与 evaluate.py 一致）: %s", official)
logger.info("行级 score_rows 对照（应接近官方分）: %s", rows_metric)